# Implicit Solver Setup

## Overview

With <code style="color:#CE9178">numerics.solver: implicit</code>, the `implicit_solver` section of the input file controls **what is solved** (equations, physical effects, stabilization) and **how it is solved** (Newton iteration, linear solver). All other sections (`grid`, `geometry`, `properties`, ...) are shared with the other solvers and are not repeated here.

All keys have defaults, but the section itself must be present with at least one key (e.g. `max_it`).

The options are grouped by the questions you have to answer when setting up a simulation:

| Step | Question | Section | Remarks |
|-|-|-|-|
| 1 | Which equations are solved? | `equations` | Mass and momentum equations are always included. **Cavitation** via complementarity condition and **energy conservation** (future version) are optional. |
| 2 | Which physical effects are included? | `physics` | Controls which terms of the above selected equations are activated. |
| 3 | Numerical stabilization required? | `stabilization` | See below for explanation. |
| 4 | How is the linearized system solved? | top level of `implicit_solver` | Direct or iterative solver, stopping criteria. |

The sections below follow this order and show details of the respective sections. After that, examples, defaults, and common set-ups are shown. Finally, it is shown how pressure boundary conditions can be applied to avoid the necessity of manual pressure/density calculations.

## Equations

Mass and momentum conservation are always solved, for the unknowns mass flux $(j_x, j_y)$ and pressure $p$.

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `cavitation` | `bool` | `False` | Elrod-Adams type cavitation via complementarity condition. Adds the cavity fraction $\theta$ ($\theta=0$: full film, $\theta>0$: cavitated) as unknown and the Fischer-Burmeister complementarity condition as equation. Requires `properties.p_cav` and `rho_l` (defaults to  EOS at `p_cav`). |

Note that cavitation can also be introduced by the equation of state, such as the Bayada-Chupin formulation. In this case keep `cavitation: False`. Stabilization is, however, still needed in most cases.

Resulting unknowns and residuals:

| Active | Unknowns | Residuals |
|--------|----------|-----------|
| always | $j_x$, $j_y$, $p$ | momentum x, momentum y, mass |
| `cavitation` | + $\theta$ | + complementarity |
| `cavitation` and `stabilization.oss` | + $\xi$ | + OSS projection |

The mass flux lives on the quadratic (Q2) and all other unknowns on the linear (Q1) grid.

## Physics

Controls which terms of the equations above are activated. Mass conservation and the pressure gradient are always included.

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `gap_shear` | `bool` | `True` | Gap-averaged wall shear stress (Poiseuille/Couette). Switching it off implies a plug flow profile. |
| `plane_shear` | `bool` | `False` | In-plane viscous stress, usually negligible in thin-film flows. |
| `inertia` | `bool` | `False` | Convective momentum transport, usually negligible in thin-film flows. |
| `body_force` | `bool` | `False` | External body force, taken from `properties.force_x`, `properties.force_y`. |
| `squeeze` | `bool` | `False` | Height rate of change $\partial h/\partial t$ as source in mass and momentum. Needed for time-dependent gap heights, e.g. elastic deformation or force balance. |
| `einm` | `bool` | `False` | Effective influence Newton method: includes the elastic influence function in the Jacobian. Only active with `properties.elastic`. |

## Stabilization

Under cavitation and in high-viscosity (high pressure with piezoviscosity) regimes, the mass conservation equation needs to be numerically stabilized. When the pressure solution shows spurious oscillations, which may lead to divergence, stabilization should be added.

Two methods are implemented. The first method, <code style="color:#6A9955">Orthogonal Subgrid Scales</code> (OSS) tackles only cavitation regimes. The second method, <code style="color:#6A9955">Upwinding Petrov-Galerkin</code> (UPG) tackles both regimes by introducing upwind bias informed by flow direction. While the latter method generally exhibits greater robustness, care should be taken to ensure that the introduced numerical diffusion does not significantly affect the solution, for example through a mesh-sensitivity analysis.

If cavitation is introduced via the equation of state, use `UPG`.

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `oss` | `bool` | `False` | Orthogonal subscale stabilization. Adds the projection $\xi$ as unknown. Only applies with `cavitation: True`. |
| `oss_factor` | `float` | `1.0` | Scaling of the OSS stabilization parameter [-]. |
| `upg` | `bool` | `False` | UPG stabilization of the mass equation. |
| `upg_factor` | `float` | `1.0` | Scaling of the UPG stabilization parameter [-]. |

## Newton and Linear Solver

The nonlinear system of every time step is solved with a Newton iteration. Each iteration assembles the Jacobian and the residual and solves a linear system for the update $\Delta q$. The defaults work for most cases; the keys below are for tuning convergence.

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `max_it` | `int` | `100` | Maximum Newton iterations per time step. |
| `tol_abs_resnorm` | `float` | `1e-6` | Absolute tolerance on the global residual norm (see below). |
| `newton_relax` | `float` | `1.0` | Newton update relaxation factor $\alpha$ in $q \leftarrow q + \alpha\,\Delta q$ [-]. Values < 1 damp the update. |
| `linear_solver` | `str` | `direct` | Linear solver for the Newton system: `direct` or `iterative`. |

**Residual tolerance:** `tol_abs_resnorm` is compared to the L2 norm of the global (unscaled) residual vector, summed over all MPI ranks. The criterion is purely absolute and at least one Newton update is performed per time step. With `tol_abs_resnorm: 0.` the criterion is never met, so every step runs `max_it` iterations. The norm of the last Newton iteration is printed in the `|R| Newton` column of the progress output, which helps in choosing a sensible value. It is not to be confused with `numerics.tol`, the steady-state criterion on the kinetic-energy residual between time steps.

**Linear solver:**
- `direct`: MUMPS LU via PETSc, `splu` via SciPy (serial only).
- `iterative`: BiCGStab via PETSc, GMRES via SciPy; relative tolerance `1e-8`.
- PETSc is required for MPI-parallel runs.

## An Example

A short configuration for a cavitating bearing with inertia term activated. Note that <code style="color:#CE9178">numerics.solver</code> needs to be set to <code style="color:#CE9178">implicit</code> to activate the implicit solver, while its settings are specified in a dedicated section:

```yaml
numerics:
    solver: implicit

implicit_solver:
    equations:
        cavitation: True
    physics:
        inertia: True
```

Everything not listed keeps its default.

## Defaults

All options with their default values. The default use case is a steady-state hydrodynamic lubrication case without cavitation.

```yaml
implicit_solver:
    max_it: 100
    tol_abs_resnorm: 1e-6
    newton_relax: 1.0
    linear_solver: direct
    equations:
        cavitation: False
    physics:
        gap_shear: True
        plane_shear: False
        inertia: False
        body_force: False
        squeeze: False
        einm: False
    stabilization:
        oss: False
        oss_factor: 1.0
        upg: False
        upg_factor: 1.0
```

## Common Set-Ups

| Goal | Settings |
|------|----------|
| Steady-state lubrication without cavitation | defaults |
| Steady-state with cavitation | `equations.cavitation: True` and stabilization |
| Squeeze with rigid walls | `physics.squeeze: True` |
| Squeeze with elastic walls | `physics.squeeze: True`, `properties.elastic`, `physics.einm: True` |
| Negative Squeeze with elastic walls | `equations.cavitation: True` and stabilization , `physics.squeeze: True`, `properties.elastic`, `physics.einm: True` |
| Steady-State Elastohydrodynamic Contact | `equations.cavitation: True` and stabilization, `properties.elastic`, `physics.einm: True` |
| Fluid driven by a force | `physics.body_force: True` and `properties.force_x`/`force_y` |

## Pressure Boundary Conditions

Dirichlet boundaries of the `grid` section are usually set by density (`xW_D`, `xE_D`, `yS_D`, `yN_D`). As an alternative, the implicit solver accepts the pressure directly. The pressure is converted to density with the equation of state.

| Field | Type | Description |
|-------|------|-------------|
| `xW_P`, `xE_P` | `float` | Pressure [Pa] at the west/east boundary. Alternative to `xW_D`, `xE_D`. |
| `yS_P`, `yN_P` | `float` | Pressure [Pa] at the south/north boundary. Alternative to `yS_D`, `yN_D`. |

```yaml
grid:
    xW: ['D', 'N', 'N']
    xE: ['D', 'N', 'N']
    xW_P: 3.36e5
    xE_P: 0.
```

- Per boundary, specify either the `_D` or the `_P` value, not both.
- A single value sets the density/pressure only. If the mass flux is also Dirichlet at that boundary, use the `_D` form with a list of values.
- With cavitation, the boundary is always fully filled ($\theta = 0$).
- Only available for the implicit solver.